# Deep Learning 076 — Why Is It Called "Self"-Attention?

Companion notebook to the lesson. Two questions, both answerable by running code:

1. **Why is self-attention a form of attention at all?** Because it is the same three
   equations — and we verify that by writing `attend(Q, K, V)` **once** and calling it
   both ways.
2. **Why "self"?** Because the scores are computed within one sequence instead of between
   two. That changes the score matrix from rectangular to square, and a square matrix has
   a diagonal — with consequences that are measurable.

| Claim | Number |
|---|---|
| self-attention vs cross-attention fed the same sequence | max difference **exactly 0.0** |
| diagonal is the row maximum, no `W_q`/`W_k` | **100.0%** of rows |
| …with `W_q`/`W_k` | 12.6% — chance |
| `X @ X.T` is symmetric | max &#124;S − Sᵀ&#124; = **0.0** |
| tied key/value on a retrieval task | MSE 0.2481 at **98.4%** retrieval |
| untied | MSE **0.0000** |

`numpy` throughout; the final experiment uses `torch`.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def unit(m):
    return m / np.linalg.norm(m, axis=-1, keepdims=True)

# The three equations, written ONCE. Nothing in here knows or cares whether
# Q came from the same sequence as K and V.
def attend(Q, K, V):
    scores = Q @ K.T / np.sqrt(K.shape[1])    # 3. dot-product alignment score
    weights = softmax(scores)                 # 2. softmax into weights
    return weights @ V, weights, scores       # 1. weighted sum

## Part A — One function, two call sites

Old attention (Luong, dot form) scores a **decoder** hidden state against every **encoder**
hidden state, then sums the encoder states with those weights:

$$c_i = \sum_j \alpha_{ij} h_j, \qquad \alpha_{ij} = \text{softmax}(e_{ij}), \qquad e_{ij} = s_i \cdot h_j$$

Self-attention scores a word's **query** against every **key**, then sums the **values**:

$$y_i = \sum_j w_{ij} v_j, \qquad w_{ij} = \text{softmax}(s_{ij}), \qquad s_{ij} = q_i \cdot k_j$$

Same three lines. So one function should serve both.

In [ ]:
n, m, d = 7, 11, 32
X = rng.normal(size=(n, d))     # one sentence
H = rng.normal(size=(m, d))     # encoder hidden states — a DIFFERENT sequence
S = rng.normal(size=(n, d))     # decoder hidden states
Wq, Wk, Wv = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3))

cross, cw, _ = attend(S, H, H)                      # Luong: h is key AND value
self_, sw, _ = attend(X @ Wq, X @ Wk, X @ Wv)       # self-attention
again, _, _ = attend(X @ Wq, X @ Wk, X @ Wv)        # the same call, spelled as "cross"

print(f"cross-attention : Q {S.shape}, K/V {H.shape} -> scores {cw.shape}")
print(f"self-attention  : Q {X.shape}, K/V {X.shape} -> scores {sw.shape}")
print(f"max |self - cross(X, X, X)| = {np.abs(self_ - again).max():.1e}")
assert np.abs(self_ - again).max() == 0.0

Not approximately zero — the same code path, so the same bits. **Self-attention is not
"like" attention. It is attention, called differently.**

The correspondence, term by term:

| old attention | self-attention | role |
|---|---|---|
| decoder state `s_i` | query `q_i` | the thing doing the asking |
| encoder state `h_j` | key `k_j` | the thing matched against |
| encoder state `h_j` **again** | value `v_j` | the thing summed |

Note the third row: old attention used **one vector for two jobs**. Part D is about
whether that matters.

## Part B — "Self" is a diagonal

A cross-attention score matrix is `n × m`. Its `(i, i)` entry compares things of different
kinds, and asking whether it equals its own transpose is not even type-correct.

A self-attention score matrix is `n × n`. It has a diagonal: **the score of every word
against itself**. Strip the projections out — as lesson 073 did — and for unit-norm
embeddings that diagonal entry is `x · x = 1`, which no cosine with another word can beat.

In [ ]:
TRIALS, n, d = 5000, 8, 64
wins_plain = wins_proj = 0
selfw_plain, selfw_proj = [], []

for _ in range(TRIALS):
    Xu = unit(rng.normal(size=(n, d)))
    _, Wp, _ = attend(Xu, Xu, Xu)                       # no parameters
    A = rng.normal(size=(d, d)) / np.sqrt(d)
    B = rng.normal(size=(d, d)) / np.sqrt(d)
    _, Wr, _ = attend(Xu @ A, Xu @ B, Xu)               # with projections
    wins_plain += (Wp.argmax(1) == np.arange(n)).sum()
    wins_proj += (Wr.argmax(1) == np.arange(n)).sum()
    selfw_plain.append(np.diag(Wp).mean())
    selfw_proj.append(np.diag(Wr).mean())

rows = TRIALS * n
print(f"diagonal is the row maximum   no W_q/W_k: {wins_plain / rows:>6.1%}"
      f"   with W_q/W_k: {wins_proj / rows:>6.1%}   (chance = {1 / n:.1%})")
print(f"mean weight a word gives itself          {np.mean(selfw_plain):.3f}"
      f"                {np.mean(selfw_proj):.3f}")

100.0% is not luck; it is an identity. **Parameterless self-attention forces every word to
attend to itself more than to anything else.**

There is a second forced constraint hiding in the same matrix.

In [ ]:
Xu = unit(rng.normal(size=(6, 64)))
_, _, Sp = attend(Xu, Xu, Xu)
A = rng.normal(size=(64, 64)) / 8
B = rng.normal(size=(64, 64)) / 8
_, _, Sr = attend(Xu @ A, Xu @ B, Xu)

print(f"max |S - S.T|   no projections : {np.abs(Sp - Sp.T).max():.1e}")
print(f"max |S - S.T|   with Q/K       : {np.abs(Sr - Sr.T).max():.3f}")
assert np.abs(Sp - Sp.T).max() == 0.0

`X @ X.T` **is** its own transpose, so word *i* attends to word *j* exactly as much as *j*
attends to *i*. Language does not work that way — an adjective depends on its noun far
more than the noun depends on the adjective.

Both constraints are artefacts of scoring a sequence **against itself**, and both die the
moment `Q ≠ K`: the score matrix becomes `X (Wq Wk.T) X.T`, and `Wq Wk.T` is not symmetric.

**`W_q` and `W_k` are what free a word from itself.**

## Part C — What "self" costs, and what it buys

Self-attention is the most expensive member of the attention family in parameters.

In [ ]:
D, VOCAB = 512, 50_000
for name, p in (("Luong (dot)", 0),
                ("Luong (general)", D * D),
                ("Bahdanau (additive)", 2 * D * D + D),
                ("Self-attention  W_q,W_k,W_v", 3 * D * D),
                (f"output layer over {VOCAB:,} words", VOCAB * D)):
    print(f"  {name:<32}{p:>12,}")
print(f"\nthe vocabulary layer is {VOCAB * D / (3 * D * D):.1f}x the whole attention block")

The last line is a pattern that has now appeared seven times in this course — MNIST,
LeNet, dog-vs-cat, VGG16, next-word prediction, seq2seq, and now attention. **The
interesting layer is cheap; the layer facing a large discrete space dominates the count.**

What the parameters buy is not fewer operations — it is **dependency depth**. Old attention
attends over encoder states, and producing those means running an LSTM one token at a time.

In [ ]:
import time

n_steps, batch = 30, 64
Wh = rng.normal(size=(D, D)) / np.sqrt(D)
Ux = rng.normal(size=(D, D)) / np.sqrt(D)
Wq2 = rng.normal(size=(D, D)) / np.sqrt(D)
seq = rng.normal(size=(n_steps, batch, D))

def rnn_encoder():
    h = np.zeros((batch, D))
    for t in range(n_steps):            # this loop CANNOT be parallelised
        h = np.tanh(h @ Wh + seq[t] @ Ux)
    return h

def self_attn_layer():
    flat = np.ascontiguousarray(seq.transpose(1, 0, 2)).reshape(-1, D)
    Q = (flat @ Wq2).reshape(batch, n_steps, D)
    K = (flat @ Wh).reshape(batch, n_steps, D)
    V = (flat @ Ux).reshape(batch, n_steps, D)
    return softmax(Q @ K.transpose(0, 2, 1) / np.sqrt(D)) @ V

def timeit(f, reps=7):
    f()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); f(); ts.append(time.perf_counter() - t0)
    return min(ts)

t_rnn, t_attn = timeit(rnn_encoder), timeit(self_attn_layer)
f_rnn = n_steps * 2 * (2 * batch * D * D)
f_att = 3 * (2 * batch * n_steps * D * D) + 2 * (2 * batch * n_steps * n_steps * D)
print(f"{'':<28}{'depth':>7}{'GFLOP':>8}{'ms':>9}{'GFLOP/s':>10}")
print(f"{'30-step recurrent encoder':<28}{n_steps:>7}{f_rnn / 1e9:>8.2f}"
      f"{t_rnn * 1e3:>9.1f}{f_rnn / 1e9 / t_rnn:>10.1f}")
print(f"{'one self-attention layer':<28}{1:>7}{f_att / 1e9:>8.2f}"
      f"{t_attn * 1e3:>9.1f}{f_att / 1e9 / t_attn:>10.1f}")

Read the **depth** column, not the milliseconds. Depth is exact — 30 against 1, and
sentence-length against 1 for any sentence — and it is a property of the computation graph
rather than of your laptop. The wall clock here is numpy on one CPU and will move around
between runs; expect attention to come out modestly ahead *while doing about 1.56× more
arithmetic*, because thirty steps that wait for each other leave the machine idle.

## Part D — The one genuine difference: key and value come apart

Old attention used `h_j` as both the thing matched against and the thing summed. Does
splitting them into `k_j` and `v_j` actually buy anything?

Build a task where the two jobs **disagree**. Every token is `[tag | payload]`. The query
carries a tag and no payload. The answer is the *payload of the token whose tag matches* —
so the matching information and the content sit in different parts of the input, on
purpose.

In [ ]:
import torch

D_TOK, N_TAG, N_PAY, SEQ = 24, 8, 16, 6
g = torch.Generator().manual_seed(1)

def batch_of(bs):
    tags = torch.zeros(bs, SEQ, dtype=torch.long)
    for b in range(bs):
        tags[b, 1:] = torch.randperm(N_TAG, generator=g)[:SEQ - 1]     # unique tags
        tags[b, 0] = tags[b, 1 + torch.randint(0, SEQ - 1, (1,), generator=g).item()]
    pay = torch.randn(bs, SEQ, N_PAY, generator=g)
    pay[:, 0] = 0.0                                                    # query has no payload
    x = torch.cat([torch.nn.functional.one_hot(tags, N_TAG).float(), pay], -1)
    match = (tags[:, 1:] == tags[:, :1])
    return x, (pay[:, 1:] * match.unsqueeze(-1)).sum(1), match.float().argmax(1)

In [ ]:
def train(mode, steps=2000, bs=256):
    torch.manual_seed(7)
    Wq = torch.randn(D_TOK, N_PAY, requires_grad=True)
    Wk = torch.randn(D_TOK, N_PAY, requires_grad=True)
    Wv = torch.randn(D_TOK, N_PAY, requires_grad=True)
    Wo = torch.randn(N_PAY, N_PAY, requires_grad=True)
    with torch.no_grad():
        for t in (Wq, Wk, Wv, Wo):
            t /= np.sqrt(D_TOK)
    params = {"tied": [Wq, Wk], "tied+out": [Wq, Wk, Wo], "untied": [Wq, Wk, Wv]}[mode]
    opt = torch.optim.Adam(params, lr=0.02)

    def forward(x):
        q, ctx = x[:, :1] @ Wq, x[:, 1:]
        k = ctx @ Wk
        v = k if mode.startswith("tied") else ctx @ Wv      # <- the whole experiment
        a = torch.softmax((q @ k.transpose(1, 2) / np.sqrt(N_PAY)).squeeze(1), -1)
        y = (a.unsqueeze(1) @ v).squeeze(1)
        return (y @ Wo if mode == "tied+out" else y), a

    for _ in range(steps):
        x, target, _ = batch_of(bs)
        y, _ = forward(x)
        loss = (y - target).pow(2).mean()
        opt.zero_grad(); loss.backward(); opt.step()

    mses, accs = [], []
    with torch.no_grad():
        for _ in range(20):
            x, target, right = batch_of(512)
            y, a = forward(x)
            mses.append((y - target).pow(2).mean().item())
            accs.append((a.argmax(1) == right).float().mean().item())
    return np.mean(mses), np.mean(accs), sum(p.numel() for p in params)

floor = np.mean([batch_of(512)[1].pow(2).mean().item() for _ in range(20)])
print(f"predicting zeros scores MSE {floor:.4f} — the do-nothing floor\n")
print(f"{'':<46}{'params':>7}{'MSE':>9}{'retrieval':>11}")
for mode, label in (("tied", "tied      key IS the value (old attention)"),
                    ("tied+out", "tied+out  one projection AFTER the sum"),
                    ("untied", "untied    separate W_v (self-attention)")):
    mse, acc, npar = train(mode)
    print(f"{label:<46}{npar:>7}{mse:>9.4f}{acc:>10.1%}")

Row one is the informative failure: **the tied model finds the right token almost every
time and still cannot say what is in it.** One 16-dimensional vector per token had to be
both matchable by tag and equal to the payload, and those are different directions in the
input, so it did the first job and abandoned the second. Retrieval and reconstruction come
apart.

Row two is the control worth staring at. The obvious patch — bolt a projection onto the
end — does not rescue it, and the algebra says why:

$$\Big(\sum_j w_j x_j W_k\Big) W_o = \sum_j w_j x_j (W_k W_o)$$

The value map is **forced to factor through `W_k`**. Whatever `W_k` discarded in order to
make the tag matching sharp is already gone before `W_o` ever sees it, and no later matrix
recovers it. Only an independent `W_v` reads the token again from scratch.

**The separation has to happen before the weighted sum, not after it.**

## The answer, both halves

- **It is attention** — the same three equations, verified as the identical function to 0.0.
- **It is "self"** — the scores are computed within one sequence rather than between two:
  intra-sequence, not inter-sequence.
- **What that costs**: 786,432 parameters, plus a square score matrix whose forced diagonal
  and forced symmetry have to be broken by `W_q` and `W_k`.
- **What it buys**: dependency depth 1 instead of 30, and a value vector that no longer has
  to double as the key.

## Try it yourself

1. In Part B, replace the random `A`, `B` with `A = B`. Does the diagonal start winning
   again? Should it?
2. Make the payload correlate with the tag (say, add `0.5 *` the one-hot to the payload) and
   re-run Part D. At what correlation does tying stop hurting?
3. Give the tied model a *bigger* `W_k` (project to 24 dims instead of 16) and see whether
   extra capacity fixes it. Predict the answer before running it.
4. Time Part C with `n_steps = 5` and `n_steps = 100`. Which column changes proportionally,
   and which one does not?